# Internal and External Validation

This notebook accompanies the QSAR tutorial chapter: **Internal and External Validation**.

## Prepare data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors

def find_example_dataset():
    candidates = [
        Path("../data/example_qsar_dataset.csv"),
        Path("data/example_qsar_dataset.csv"),
        Path("../../data/example_qsar_dataset.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find example_qsar_dataset.csv.")

def mol_from_smiles(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def calculate_all_rdkit_descriptors(mol):
    return Descriptors.CalcMolDescriptors(mol, missingVal=np.nan)

# Load data
df = pd.read_csv(find_example_dataset())

# Convert SMILES to RDKit molecules and remove invalid molecules
df["Mol"] = df["SMILES"].apply(mol_from_smiles)
df_clean = df.dropna(subset=["Mol"]).copy()

# Calculate all default RDKit descriptors
descriptor_rows = [
    calculate_all_rdkit_descriptors(mol)
    for mol in df_clean["Mol"]
]

X = pd.DataFrame(descriptor_rows, index=df_clean.index)

# Convert to numeric and remove descriptor columns containing missing values
X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1)

# Target values
y = df_clean["Activity"].astype(float)

print("Number of valid molecules:", len(df_clean))
print("Number of RDKit descriptors:", X.shape[1])

X.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
import pandas as pd

# Basic descriptor filtering
selector = VarianceThreshold(threshold=1e-8)
X_var = pd.DataFrame(
    selector.fit_transform(X),
    columns=X.columns[selector.get_support()],
    index=X.index
)

# Use top 30 descriptors by variance for faster demonstration
top_features = X_var.var().sort_values(ascending=False).head(30).index
X_model = X_var[top_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42
)

print("Training set:", X_train.shape)
print("External test set:", X_test.shape)

## 5-fold cross-validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="r2"
)

print("5-fold CV R2 scores:", cv_scores)
print("Mean CV R2:", np.mean(cv_scores))
print("Standard deviation:", np.std(cv_scores))

## Leave-one-out cross-validation

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

loo = LeaveOneOut()

y_pred_loo = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=loo
)

r2_loo = r2_score(y_train, y_pred_loo)
mae_loo = mean_absolute_error(y_train, y_pred_loo)
rmse_loo = np.sqrt(mean_squared_error(y_train, y_pred_loo))

print("Number of LOOCV rounds:", len(y_pred_loo))
print("LOOCV R2:", r2_loo)
print("LOOCV MAE:", mae_loo)
print("LOOCV RMSE:", rmse_loo)

## External validation

In [ ]:
final_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

final_model.fit(X_train, y_train)

y_pred_train = final_model.predict(X_train)
y_pred_test = final_model.predict(X_test)

print("Training R2:", r2_score(y_train, y_pred_train))
print("External test R2:", r2_score(y_test, y_pred_test))
print("External test MAE:", mean_absolute_error(y_test, y_pred_test))
print("External test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred_test)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.xlabel("Experimental activity")
plt.ylabel("Predicted activity")
plt.title("External Validation: Predicted vs. Experimental")
plt.show()